In [ ]:
# =========================================================
# EPOCH Fever Day - Final Integrated Code
# Taiwanese Bankruptcy Prediction
# Metric: F2 Score
#
# 구성
# 1) 데이터 로드
# 2) 컬럼명 정리
# 3) 기본 전처리
# 4) 파생변수 생성
# 5) 이상치 clipping
# 6) 중요도 기반 feature selection
# 7) LGBM + XGB seed ensemble (Stratified 5-Fold CV)
# 8) OOF 기반 threshold 최적화
# 9) test 예측 및 제출 파일 저장
#
# 제출 파일: result.csv
# 컬럼: ID, Bankrupt?
# =========================================================

import os
import re
import gc
import random
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import fbeta_score
from sklearn.impute import SimpleImputer

from lightgbm import LGBMClassifier
from xgboost import XGBClassifier


warnings.filterwarnings("ignore")

In [ ]:
# =========================================================
# 0. 환경 설정
# =========================================================
RANDOM_STATE = 42
N_SPLITS = 5
SEEDS = [42, 52, 62]

TOP_N_FEATURES = 55
CLIP_QUANTILE_LOW = 0.005
CLIP_QUANTILE_HIGH = 0.995

SUBMISSION_DIR = Path("./submission")
SUBMISSION_DIR.mkdir(exist_ok=True)

TRAIN_PATH_CANDIDATES = [
    "./train.parquet",
    "/mnt/data/train.parquet",
    "train.parquet"
]

TEST_PATH_CANDIDATES = [
    "./test.parquet",
    "/mnt/data/test.parquet",
    "test.parquet"
]

In [ ]:
# =========================================================
# 1. 유틸 함수
# =========================================================
def seed_everything(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)


def find_existing_path(candidates):
    for p in candidates:
        if os.path.exists(p):
            return p
    raise FileNotFoundError(f"파일을 찾지 못했습니다. 후보 경로: {candidates}")


def sanitize_column_name(col: str) -> str:
    col = str(col).strip()
    col = col.replace("\n", " ")
    col = re.sub(r"\s+", " ", col)
    return col


def make_safe_columns(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df.columns = [sanitize_column_name(c) for c in df.columns]
    return df


def safe_divide(a, b):
    return a / (b.replace(0, np.nan) + 1e-12)


def f2_score_eval(y_true, y_pred):
    return fbeta_score(y_true, y_pred, beta=2)


def evaluate_thresholds(y_true, proba, thresholds):
    rows = []
    for th in thresholds:
        pred = (proba >= th).astype(int)
        f2 = f2_score_eval(y_true, pred)
        rows.append({
            "threshold": th,
            "f2": f2,
            "predicted_positive": int(pred.sum())
        })
    return pd.DataFrame(rows).sort_values(["f2", "threshold"], ascending=[False, True]).reset_index(drop=True)

In [ ]:
# =========================================================
# 2. 데이터 로드
# =========================================================
seed_everything(RANDOM_STATE)

train_path = find_existing_path(TRAIN_PATH_CANDIDATES)
test_path = find_existing_path(TEST_PATH_CANDIDATES)

train = pd.read_parquet(train_path)
test = pd.read_parquet(test_path)

train = make_safe_columns(train)
test = make_safe_columns(test)

print("=" * 70)
print("Loaded files")
print(f"train path: {train_path}")
print(f"test path : {test_path}")
print(f"train shape: {train.shape}")
print(f"test shape : {test.shape}")
print("=" * 70)

In [ ]:
# =========================================================
# 3. 타겟 / ID 분리
# =========================================================
TARGET_COL = "Bankrupt?"
ID_COL = "ID"

if TARGET_COL not in train.columns:
    raise ValueError(f"train 데이터에 타겟 컬럼 '{TARGET_COL}' 이 없습니다.")

if ID_COL not in train.columns or ID_COL not in test.columns:
    raise ValueError(f"ID 컬럼 '{ID_COL}' 이 train/test 중 하나에 없습니다.")

y = train[TARGET_COL].astype(int).copy()

X = train.drop(columns=[TARGET_COL]).copy()
X_test = test.copy()

train_ids = X[ID_COL].copy()
test_ids = X_test[ID_COL].copy()

X = X.drop(columns=[ID_COL])
X_test = X_test.drop(columns=[ID_COL])

print(f"feature count (raw): {X.shape[1]}")
print(f"target positive: {int(y.sum())}, negative: {int((1-y).sum())}")
print(f"positive ratio : {y.mean():.4f}")

In [ ]:
# =========================================================
# 4. 파생변수 생성
#    - 기존에 자주 잘 먹히던 재무 안정성 / 수익성 / 부채 대비 계열 위주
#    - 컬럼이 없는 경우는 자동 스킵
# =========================================================
def add_engineered_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    # 자주 쓰는 후보 컬럼 사전
    col = {c: c for c in df.columns}

    # 짧게 참조용 함수
    def has(*cols):
        return all(c in df.columns for c in cols)

    # -----------------------------------------------------
    # 수익성 관련
    # -----------------------------------------------------
    roa_cols = [
        "ROA(A) before interest and % after tax",
        "ROA(B) before interest and depreciation after tax",
        "ROA(C) before interest and depreciation before interest"
    ]
    existing_roa_cols = [c for c in roa_cols if c in df.columns]

    if len(existing_roa_cols) >= 2:
        df["feat_roa_mean"] = df[existing_roa_cols].mean(axis=1)
        df["feat_roa_std"] = df[existing_roa_cols].std(axis=1)
        df["feat_roa_min"] = df[existing_roa_cols].min(axis=1)
        df["feat_roa_max"] = df[existing_roa_cols].max(axis=1)

    if has("Operating Gross Margin", "Realized Sales Gross Margin"):
        df["feat_margin_gap"] = df["Operating Gross Margin"] - df["Realized Sales Gross Margin"]

    if has("Net Income to Total Assets", "Total Asset Turnover"):
        df["feat_income_asset_turnover"] = (
            df["Net Income to Total Assets"] * df["Total Asset Turnover"]
        )

    # -----------------------------------------------------
    # 부채 / 안정성 관련
    # -----------------------------------------------------
    if has("Debt ratio %", "Current Ratio"):
        df["feat_liquidity_vs_debt"] = safe_divide(df["Current Ratio"], df["Debt ratio %"])

    if has("Debt ratio %", "Quick Ratio"):
        df["feat_quick_vs_debt"] = safe_divide(df["Quick Ratio"], df["Debt ratio %"])

    if has("Debt ratio %", "Cash/Current Liability"):
        df["feat_cash_vs_debt"] = safe_divide(df["Cash/Current Liability"], df["Debt ratio %"])

    if has("Total debt/Total net worth", "Net worth/Assets"):
        df["feat_networth_vs_leverage"] = safe_divide(
            df["Net worth/Assets"], df["Total debt/Total net worth"]
        )

    if has("Retained Earnings to Total Assets", "Debt ratio %"):
        df["feat_retained_vs_debt"] = safe_divide(
            df["Retained Earnings to Total Assets"], df["Debt ratio %"]
        )

    if has("Borrowing dependency", "Cash Flow to Total Assets"):
        df["feat_borrowing_cashflow_gap"] = (
            df["Cash Flow to Total Assets"] - df["Borrowing dependency"]
        )

    if has("Liability to Equity", "Equity to Liability"):
        df["feat_equity_liability_balance"] = (
            df["Equity to Liability"] - df["Liability to Equity"]
        )

    # -----------------------------------------------------
    # 영업 / 현금흐름 관련
    # -----------------------------------------------------
    if has("Continuous interest rate (after tax)", "Interest Expense Ratio"):
        df["feat_interest_cover_proxy"] = safe_divide(
            df["Continuous interest rate (after tax)"], df["Interest Expense Ratio"]
        )

    if has("Cash Flow to Sales", "Net Value Growth Rate"):
        df["feat_cashflow_growth_mix"] = (
            df["Cash Flow to Sales"] * df["Net Value Growth Rate"]
        )

    if has("Current Liability to Assets", "Cash/Current Liability"):
        df["feat_cash_liability_pressure"] = safe_divide(
            df["Cash/Current Liability"], df["Current Liability to Assets"]
        )

    if has("Inventory/Current Liability", "Current Liability to Assets"):
        df["feat_inventory_liability_pressure"] = safe_divide(
            df["Inventory/Current Liability"], df["Current Liability to Assets"]
        )

    if has("Working Capital to Total Assets", "Current Liability to Assets"):
        df["feat_workingcap_liability_gap"] = (
            df["Working Capital to Total Assets"] - df["Current Liability to Assets"]
        )

    # -----------------------------------------------------
    # 성장성 관련
    # -----------------------------------------------------
    growth_cols = [
        "Net Income Growth Rate",
        "Operating Profit Growth Rate",
        "Total Asset Growth Rate",
        "Total Equity Growth Rate"
    ]
    existing_growth_cols = [c for c in growth_cols if c in df.columns]

    if len(existing_growth_cols) >= 2:
        df["feat_growth_mean"] = df[existing_growth_cols].mean(axis=1)
        df["feat_growth_std"] = df[existing_growth_cols].std(axis=1)

    if has("Net Value Growth Rate", "Total Asset Growth Rate"):
        df["feat_value_asset_growth_gap"] = (
            df["Net Value Growth Rate"] - df["Total Asset Growth Rate"]
        )

    # -----------------------------------------------------
    # 로그/안정화 변환
    # -----------------------------------------------------
    log_candidates = [
        "Interest Expense Ratio",
        "Total Asset Turnover",
        "Accounts Receivable Turnover",
        "Inventory Turnover Rate (times)",
        "Cash Turnover Rate",
        "Net Value Growth Rate",
        "Operating Profit Growth Rate",
        "Total Asset Growth Rate"
    ]

    for c in log_candidates:
        if c in df.columns:
            # 값 범위가 0~1일 수도 있고 이상치가 있을 수도 있으니 abs + log1p
            df[f"log_abs_{c}"] = np.log1p(np.abs(df[c]))

    # -----------------------------------------------------
    # 상호작용
    # -----------------------------------------------------
    if has("Net Income to Total Assets", "Borrowing dependency"):
        df["feat_profit_vs_borrowing"] = (
            df["Net Income to Total Assets"] - df["Borrowing dependency"]
        )

    if has("ROA(C) before interest and depreciation before interest", "Debt ratio %"):
        df["feat_roac_vs_debt"] = safe_divide(
            df["ROA(C) before interest and depreciation before interest"],
            df["Debt ratio %"]
        )

    if has("ROA(A) before interest and % after tax", "Current Liability to Assets"):
        df["feat_roaa_vs_current_liability"] = safe_divide(
            df["ROA(A) before interest and % after tax"],
            df["Current Liability to Assets"]
        )

    return df


X = add_engineered_features(X)
X_test = add_engineered_features(X_test)

print(f"feature count (after FE): {X.shape[1]}")

In [ ]:
# =========================================================
# 5. train/test 컬럼 정렬 맞추기
# =========================================================
missing_in_test = [c for c in X.columns if c not in X_test.columns]
missing_in_train = [c for c in X_test.columns if c not in X.columns]

if missing_in_test:
    for c in missing_in_test:
        X_test[c] = np.nan

if missing_in_train:
    X_test = X_test.drop(columns=missing_in_train)

X_test = X_test[X.columns].copy()

print(f"Aligned feature count: {X.shape[1]} / {X_test.shape[1]}")

In [ ]:
# =========================================================
# 6. 결측치 처리
# =========================================================
imputer = SimpleImputer(strategy="median")
X_imputed = pd.DataFrame(imputer.fit_transform(X), columns=X.columns, index=X.index)
X_test_imputed = pd.DataFrame(imputer.transform(X_test), columns=X_test.columns, index=X_test.index)

In [ ]:
# =========================================================
# 7. clipping 정보 생성
#    - train 기준 quantile clipping
# =========================================================
clip_info = {}
for c in X_imputed.columns:
    lower = X_imputed[c].quantile(CLIP_QUANTILE_LOW)
    upper = X_imputed[c].quantile(CLIP_QUANTILE_HIGH)
    clip_info[c] = (lower, upper)

for c in X_imputed.columns:
    lower, upper = clip_info[c]
    X_imputed[c] = X_imputed[c].clip(lower, upper)
    X_test_imputed[c] = X_test_imputed[c].clip(lower, upper)

In [ ]:
# =========================================================
# 8. 중요도 기반 feature selection
#    - 먼저 빠르게 LGBM으로 중요도 측정
# =========================================================
def get_feature_importance_lgbm(X_df, y_series, n_splits=5, seed=42):
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    importances = pd.DataFrame({"feature": X_df.columns, "importance": 0.0})

    for fold, (tr_idx, va_idx) in enumerate(skf.split(X_df, y_series), 1):
        X_tr, X_va = X_df.iloc[tr_idx], X_df.iloc[va_idx]
        y_tr, y_va = y_series.iloc[tr_idx], y_series.iloc[va_idx]

        model = LGBMClassifier(
            n_estimators=700,
            learning_rate=0.03,
            num_leaves=31,
            max_depth=6,
            min_child_samples=30,
            subsample=0.9,
            colsample_bytree=0.9,
            reg_alpha=0.01,
            reg_lambda=1.0,
            objective="binary",
            random_state=seed + fold,
            n_jobs=-1,
            class_weight="balanced",
            verbosity=-1
        )

        model.fit(X_tr, y_tr)
        importances["importance"] += model.feature_importances_ / n_splits

    importances = importances.sort_values("importance", ascending=False).reset_index(drop=True)
    return importances


feature_importance_df = get_feature_importance_lgbm(X_imputed, y, n_splits=N_SPLITS, seed=RANDOM_STATE)
selected_features = feature_importance_df["feature"].head(TOP_N_FEATURES).tolist()

print("=" * 70)
print(f"Selected top {TOP_N_FEATURES} features")
print(feature_importance_df.head(20))
print("=" * 70)

X_sel = X_imputed[selected_features].copy()
X_test_sel = X_test_imputed[selected_features].copy()

In [ ]:
# =========================================================
# 9. 모델 파라미터
#    - 너무 과하게 복잡한 튜닝보다,
#      지금까지 흐름상 안정적으로 잘 나올 만한 세팅으로 정리
# =========================================================
def get_lgbm_params(seed):
    return {
        "n_estimators": 1100,
        "learning_rate": 0.018,
        "num_leaves": 29,
        "max_depth": 6,
        "min_child_samples": 38,
        "subsample": 0.97,
        "colsample_bytree": 0.90,
        "reg_alpha": 0.012,
        "reg_lambda": 1.08,
        "objective": "binary",
        "random_state": seed,
        "n_jobs": -1,
        "class_weight": "balanced",
        "verbosity": -1
    }


def get_xgb_params(seed, pos_weight):
    return {
        "n_estimators": 900,
        "learning_rate": 0.02,
        "max_depth": 5,
        "min_child_weight": 3,
        "subsample": 0.90,
        "colsample_bytree": 0.85,
        "gamma": 0.10,
        "reg_alpha": 0.05,
        "reg_lambda": 1.5,
        "objective": "binary:logistic",
        "eval_metric": "logloss",
        "random_state": seed,
        "n_jobs": -1,
        "scale_pos_weight": pos_weight,
        "tree_method": "hist"
    }

In [ ]:
# =========================================================
# 10. CV + OOF + test prediction
#     - LGBM / XGB 둘 다 seed ensemble
#     - fold별 OOF 저장
# =========================================================
def run_cv_ensemble(X_df, y_series, X_test_df, seeds, n_splits=5):
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=RANDOM_STATE)

    oof_lgb = np.zeros(len(X_df))
    oof_xgb = np.zeros(len(X_df))

    test_lgb = np.zeros(len(X_test_df))
    test_xgb = np.zeros(len(X_test_df))

    pos_weight = (len(y_series) - y_series.sum()) / y_series.sum()

    total_runs = 0

    for seed in seeds:
        print(f"\n[Seed {seed}] start")
        seed_everything(seed)

        for fold, (tr_idx, va_idx) in enumerate(skf.split(X_df, y_series), 1):
            X_tr, X_va = X_df.iloc[tr_idx], X_df.iloc[va_idx]
            y_tr, y_va = y_series.iloc[tr_idx], y_series.iloc[va_idx]

            # ---------------------------
            # LightGBM
            # ---------------------------
            lgb_model = LGBMClassifier(**get_lgbm_params(seed + fold))
            lgb_model.fit(X_tr, y_tr)

            va_pred_lgb = lgb_model.predict_proba(X_va)[:, 1]
            te_pred_lgb = lgb_model.predict_proba(X_test_df)[:, 1]

            oof_lgb[va_idx] += va_pred_lgb / len(seeds)
            test_lgb += te_pred_lgb / (len(seeds) * n_splits)

            # ---------------------------
            # XGBoost
            # ---------------------------
            xgb_model = XGBClassifier(**get_xgb_params(seed + fold, pos_weight=float(pos_weight)))
            xgb_model.fit(X_tr, y_tr)

            va_pred_xgb = xgb_model.predict_proba(X_va)[:, 1]
            te_pred_xgb = xgb_model.predict_proba(X_test_df)[:, 1]

            oof_xgb[va_idx] += va_pred_xgb / len(seeds)
            test_xgb += te_pred_xgb / (len(seeds) * n_splits)

            total_runs += 1
            print(f"  Fold {fold}/{n_splits} done")

            del lgb_model, xgb_model, X_tr, X_va, y_tr, y_va
            gc.collect()

    return oof_lgb, oof_xgb, test_lgb, test_xgb


oof_lgb, oof_xgb, test_lgb, test_xgb = run_cv_ensemble(
    X_sel, y, X_test_sel, seeds=SEEDS, n_splits=N_SPLITS
)

In [ ]:
# =========================================================
# 10. CV + OOF + test prediction
#     추가:
#     각 seed/fold/model 단위 hard label 후보도 저장
# =========================================================
def run_cv_ensemble(X_df, y_series, X_test_df, seeds, n_splits=5):
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=RANDOM_STATE)

    oof_lgb = np.zeros(len(X_df))
    oof_xgb = np.zeros(len(X_df))

    test_lgb = np.zeros(len(X_test_df))
    test_xgb = np.zeros(len(X_test_df))

    pos_weight = (len(y_series) - y_series.sum()) / y_series.sum()

    # hard voting용 raw proba 저장
    # train 쪽은 OOF 위치에만 기록
    oof_lgb_seed_list = []
    oof_xgb_seed_list = []
    test_lgb_seed_list = []
    test_xgb_seed_list = []

    for seed in seeds:
        print(f"\n[Seed {seed}] start")

        oof_lgb_seed = np.zeros(len(X_df))
        oof_xgb_seed = np.zeros(len(X_df))

        test_lgb_seed = np.zeros(len(X_test_df))
        test_xgb_seed = np.zeros(len(X_test_df))

        for fold, (tr_idx, va_idx) in enumerate(skf.split(X_df, y_series), 1):
            X_tr, X_va = X_df.iloc[tr_idx], X_df.iloc[va_idx]
            y_tr, y_va = y_series.iloc[tr_idx], y_series.iloc[va_idx]

            lgb_model = LGBMClassifier(**get_lgbm_params(seed + fold))
            lgb_model.fit(X_tr, y_tr)

            va_pred_lgb = lgb_model.predict_proba(X_va)[:, 1]
            te_pred_lgb = lgb_model.predict_proba(X_test_df)[:, 1]

            oof_lgb[va_idx] += va_pred_lgb / len(seeds)
            test_lgb += te_pred_lgb / (len(seeds) * n_splits)

            oof_lgb_seed[va_idx] = va_pred_lgb
            test_lgb_seed += te_pred_lgb / n_splits

            xgb_model = XGBClassifier(**get_xgb_params(seed + fold, pos_weight=float(pos_weight)))
            xgb_model.fit(X_tr, y_tr)

            va_pred_xgb = xgb_model.predict_proba(X_va)[:, 1]
            te_pred_xgb = xgb_model.predict_proba(X_test_df)[:, 1]

            oof_xgb[va_idx] += va_pred_xgb / len(seeds)
            test_xgb += te_pred_xgb / (len(seeds) * n_splits)

            oof_xgb_seed[va_idx] = va_pred_xgb
            test_xgb_seed += te_pred_xgb / n_splits

            print(f"  Fold {fold}/{n_splits} done")

            del lgb_model, xgb_model, X_tr, X_va, y_tr, y_va
            gc.collect()

        oof_lgb_seed_list.append(oof_lgb_seed.copy())
        oof_xgb_seed_list.append(oof_xgb_seed.copy())
        test_lgb_seed_list.append(test_lgb_seed.copy())
        test_xgb_seed_list.append(test_xgb_seed.copy())

    return {
        "oof_lgb": oof_lgb,
        "oof_xgb": oof_xgb,
        "test_lgb": test_lgb,
        "test_xgb": test_xgb,
        "oof_lgb_seed_list": oof_lgb_seed_list,
        "oof_xgb_seed_list": oof_xgb_seed_list,
        "test_lgb_seed_list": test_lgb_seed_list,
        "test_xgb_seed_list": test_xgb_seed_list
    }


cv_result = run_cv_ensemble(
    X_sel, y, X_test_sel, seeds=SEEDS, n_splits=N_SPLITS
)

oof_lgb = cv_result["oof_lgb"]
oof_xgb = cv_result["oof_xgb"]
test_lgb = cv_result["test_lgb"]
test_xgb = cv_result["test_xgb"]

oof_lgb_seed_list = cv_result["oof_lgb_seed_list"]
oof_xgb_seed_list = cv_result["oof_xgb_seed_list"]
test_lgb_seed_list = cv_result["test_lgb_seed_list"]
test_xgb_seed_list = cv_result["test_xgb_seed_list"]


# =========================================================
# 11. 단일 모델 threshold 최적화
# =========================================================
lgb_th, lgb_f2, lgb_th_table = optimize_threshold(y, oof_lgb, "LGBM")
xgb_th, xgb_f2, xgb_th_table = optimize_threshold(y, oof_xgb, "XGB")


# =========================================================
# 12. soft blend 비율 탐색
# =========================================================
blend_candidates = []
weight_grid = [0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8]

for w_lgb in weight_grid:
    w_xgb = 1.0 - w_lgb
    oof_blend = w_lgb * oof_lgb + w_xgb * oof_xgb

    temp = evaluate_thresholds(
        y_true=y,
        proba=oof_blend,
        thresholds=np.arange(0.01, 0.10, 0.001)
    ).head(1).copy()

    temp["w_lgb"] = w_lgb
    temp["w_xgb"] = w_xgb
    blend_candidates.append(temp)

blend_result = pd.concat(blend_candidates, axis=0).sort_values(
    ["f2", "threshold"], ascending=[False, True]
).reset_index(drop=True)

best_w_lgb = float(blend_result.loc[0, "w_lgb"])
best_w_xgb = float(blend_result.loc[0, "w_xgb"])

print("\n" + "=" * 70)
print("Best blend result")
print(blend_result.head(10))
print("=" * 70)

oof_blend = best_w_lgb * oof_lgb + best_w_xgb * oof_xgb
test_blend = best_w_lgb * test_lgb + best_w_xgb * test_xgb

blend_th, blend_f2, blend_th_table = optimize_threshold(y, oof_blend, "SoftBlend")


# =========================================================
# 13. hard voting 후보 만들기
#     잘 나온 것들 기반:
#     - LGB seed별 hard label
#     - XGB seed별 hard label
#     - LGB 전체 ensemble hard label
#     - XGB 전체 ensemble hard label
#     - soft blend hard label
# =========================================================
oof_binary_members = []
test_binary_members = []

# 13-1) seed별 LGB
for i, seed in enumerate(SEEDS):
    oof_bin = (oof_lgb_seed_list[i] >= lgb_th).astype(int)
    test_bin = (test_lgb_seed_list[i] >= lgb_th).astype(int)

    oof_binary_members.append(oof_bin)
    test_binary_members.append(test_bin)

# 13-2) seed별 XGB
for i, seed in enumerate(SEEDS):
    oof_bin = (oof_xgb_seed_list[i] >= xgb_th).astype(int)
    test_bin = (test_xgb_seed_list[i] >= xgb_th).astype(int)

    oof_binary_members.append(oof_bin)
    test_binary_members.append(test_bin)

# 13-3) 전체 LGB ensemble
oof_binary_members.append((oof_lgb >= lgb_th).astype(int))
test_binary_members.append((test_lgb >= lgb_th).astype(int))

# 13-4) 전체 XGB ensemble
oof_binary_members.append((oof_xgb >= xgb_th).astype(int))
test_binary_members.append((test_xgb >= xgb_th).astype(int))

# 13-5) soft blend ensemble
oof_binary_members.append((oof_blend >= blend_th).astype(int))
test_binary_members.append((test_blend >= blend_th).astype(int))

oof_binary_matrix = np.column_stack(oof_binary_members)
test_binary_matrix = np.column_stack(test_binary_members)

print("\nHard voting member count:", oof_binary_matrix.shape[1])


# =========================================================
# 14. hard voting 조건 탐색
#     총 member 수 대비 몇 표 이상이면 1로 볼지 탐색
# =========================================================
vote_results = []
n_members = oof_binary_matrix.shape[1]

for min_votes in range(2, n_members + 1):
    oof_vote_pred = majority_vote_from_binary_matrix(oof_binary_matrix, min_votes=min_votes)
    f2 = f2_score_eval(y, oof_vote_pred)

    vote_results.append({
        "min_votes": min_votes,
        "f2": f2,
        "predicted_positive": int(oof_vote_pred.sum())
    })

vote_result_df = pd.DataFrame(vote_results).sort_values(
    ["f2", "min_votes"], ascending=[False, True]
).reset_index(drop=True)

best_min_votes = int(vote_result_df.loc[0, "min_votes"])
best_vote_f2 = float(vote_result_df.loc[0, "f2"])

print("\n" + "=" * 70)
print("Hard Voting Results")
print(vote_result_df)
print("=" * 70)
print(f"Best hard voting min_votes = {best_min_votes}, F2 = {best_vote_f2:.6f}")


# =========================================================
# 15. 최종 후보 비교
# =========================================================
candidate_summary = pd.DataFrame([
    {"method": "LGBM",       "f2": lgb_f2},
    {"method": "XGB",        "f2": xgb_f2},
    {"method": "SoftBlend",  "f2": blend_f2},
    {"method": "HardVoting", "f2": best_vote_f2},
]).sort_values("f2", ascending=False).reset_index(drop=True)

print("\n" + "=" * 70)
print("Candidate Summary")
print(candidate_summary)
print("=" * 70)


# =========================================================
# 16. 최종 제출용 예측 선택
#     OOF 기준 더 좋은 쪽 선택
# =========================================================
if best_vote_f2 > blend_f2:
    final_method = "HardVoting"
    final_test_pred = majority_vote_from_binary_matrix(test_binary_matrix, min_votes=best_min_votes)
else:
    final_method = "SoftBlend"
    final_test_pred = (test_blend >= blend_th).astype(int)

print(f"\nFinal selected method: {final_method}")
print(f"Predicted positive count: {int(final_test_pred.sum())}")
print(f"Predicted negative count: {int((1 - final_test_pred).sum())}")


# =========================================================
# 17. 제출 파일 저장
# =========================================================
submission_final = pd.DataFrame({
    "ID": test_ids,
    "Bankrupt?": final_test_pred.astype(int)
})

assert list(submission_final.columns) == ["ID", "Bankrupt?"]
assert submission_final["ID"].isnull().sum() == 0
assert submission_final["ID"].duplicated().sum() == 0
assert len(submission_final) == len(test)
assert set(submission_final["Bankrupt?"].unique()).issubset({0, 1})

final_path = SUBMISSION_DIR / "result.csv"
submission_final.to_csv(final_path, index=False, encoding="utf-8")

print("\n" + "=" * 70)
print(f"Final submission saved: {final_path}")
print(submission_final.head())
print("=" * 70)


# =========================================================
# 18. 참고용: soft / hard 둘 다 따로 저장
# =========================================================
submission_soft = pd.DataFrame({
    "ID": test_ids,
    "Bankrupt?": (test_blend >= blend_th).astype(int)
})
submission_soft.to_csv(SUBMISSION_DIR / "result_soft_blend.csv", index=False, encoding="utf-8")

submission_hard = pd.DataFrame({
    "ID": test_ids,
    "Bankrupt?": majority_vote_from_binary_matrix(test_binary_matrix, min_votes=best_min_votes).astype(int)
})
submission_hard.to_csv(SUBMISSION_DIR / "result_hard_voting.csv", index=False, encoding="utf-8")

print("Saved extra files:")
print("- submission/result_soft_blend.csv")
print("- submission/result_hard_voting.csv")

In [ ]:

# =========================================================
# 11. 블렌딩 비율 탐색
#     - OOF 기준 가장 F2 잘 나오는 조합 선택
# =========================================================
blend_candidates = []
weight_grid = [0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8]

for w_lgb in weight_grid:
    w_xgb = 1.0 - w_lgb
    oof_blend = w_lgb * oof_lgb + w_xgb * oof_xgb

    temp = evaluate_thresholds(
        y_true=y,
        proba=oof_blend,
        thresholds=np.arange(0.01, 0.10, 0.001)
    ).head(1).copy()

    temp["w_lgb"] = w_lgb
    temp["w_xgb"] = w_xgb
    blend_candidates.append(temp)

blend_result = pd.concat(blend_candidates, axis=0).sort_values(
    ["f2", "threshold"], ascending=[False, True]
).reset_index(drop=True)

best_w_lgb = float(blend_result.loc[0, "w_lgb"])
best_w_xgb = float(blend_result.loc[0, "w_xgb"])

print("\n" + "=" * 70)
print("Best blend result")
print(blend_result.head(10))
print("=" * 70)

oof_blend = best_w_lgb * oof_lgb + best_w_xgb * oof_xgb
test_blend = best_w_lgb * test_lgb + best_w_xgb * test_xgb

In [ ]:

# =========================================================
# 12. threshold 탐색
#     - OOF 기준으로 최적 threshold 선택
# =========================================================
threshold_grid = np.concatenate([
    np.arange(0.010, 0.040, 0.001),
    np.arange(0.040, 0.080, 0.002),
    np.arange(0.080, 0.121, 0.005)
])

threshold_result = evaluate_thresholds(y, oof_blend, threshold_grid)

print("\n" + "=" * 70)
print("Threshold search result (top 20)")
print(threshold_result.head(20))
print("=" * 70)

best_threshold = float(threshold_result.loc[0, "threshold"])
best_oof_f2 = float(threshold_result.loc[0, "f2"])

print(f"\nBest threshold: {best_threshold:.5f}")
print(f"Best OOF F2  : {best_oof_f2:.6f}")


# =========================================================
# 13. 최종 예측
# =========================================================
test_pred = (test_blend >= best_threshold).astype(int)

print("\nPrediction summary")
print(f"Predicted positive count: {int(test_pred.sum())}")
print(f"Predicted negative count: {int((1 - test_pred).sum())}")


# =========================================================
# 14. 제출 파일 생성
# =========================================================
submission = pd.DataFrame({
    "ID": test_ids,
    "Bankrupt?": test_pred.astype(int)
})

# 검증
assert list(submission.columns) == ["ID", "Bankrupt?"]
assert submission["ID"].isnull().sum() == 0
assert submission["ID"].duplicated().sum() == 0
assert len(submission) == len(test)
assert set(submission["Bankrupt?"].unique()).issubset({0, 1})

save_path = SUBMISSION_DIR / "result.csv"
submission.to_csv(save_path, index=False, encoding="utf-8")

print("\n" + "=" * 70)
print(f"Submission saved: {save_path}")
print(submission.head())
print("=" * 70)